# 05 - Results and Error Analysis

This notebook interprets the frozen experiment and evaluation. It uses only saved predictions, reference annotations, cleaned Markdown, and the final Notebook 04 matching rules.

## 1. Setup and final result summary

In [1]:
# Import the tools needed to read the saved results and summarize errors.
from pathlib import Path
import contextlib
import io
import json
import math
import re
import sys
import unicodedata

import pandas as pd

# Find the project folder whether the notebook starts from the root or notebooks/.
PROJECT_FOLDER = Path.cwd().resolve()
for candidate in [PROJECT_FOLDER, PROJECT_FOLDER.parent]:
    if (candidate / "data/final_results/experiment_summary.csv").exists():
        PROJECT_FOLDER = candidate
        break
else:
    raise FileNotFoundError("Could not find the project data directory")

sys.path.insert(0, str(PROJECT_FOLDER / "src"))
from extraction_schema import DocumentExtraction

# Use only the frozen annotations, documents, and extraction results.
REFERENCE_PATH = PROJECT_FOLDER / "data/evaluation/manual_annotations.csv"
MANIFEST_PATH = PROJECT_FOLDER / "data/evaluation/manifest.json"
DOCUMENT_DIR = PROJECT_FOLDER / "data/evaluation/documents"
RESULTS_DIR = PROJECT_FOLDER / "data/final_results"
NOTEBOOK_04_PATH = PROJECT_FOLDER / "notebooks/04_evaluation.ipynb"

# Load the reference table, experiment summary, and document manifest.
reference = pd.read_csv(REFERENCE_PATH).fillna("")
summary = pd.read_csv(RESULTS_DIR / "experiment_summary.csv")
manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))

# Keep the metadata and cleaned text available for later examples and checks.
document_metadata = {
    item["document_id"]: item for item in manifest["documents"]
}
document_texts = {
    item["document_id"]: (DOCUMENT_DIR / item["filename"]).read_text(encoding="utf-8")
    for item in manifest["documents"]
}


# Load a final output only when the workflow completed that document.
def load_final_output(workflow, document_number, expected_document_id):
    path = RESULTS_DIR / workflow / f"document_{document_number:02d}" / "final.json"
    if not path.exists():
        return None
    output = DocumentExtraction.model_validate_json(path.read_text(encoding="utf-8"))
    assert output.document_id == expected_document_id
    return output.model_dump(mode="json")


# Organize saved predictions by workflow and document for the later analysis.
predictions = {"Workflow A": {}, "Workflow B": {}}
completion_rows = []
for index, row in summary.iterrows():
    document_number = index + 1
    document_id = row["document_id"]
    output_a = load_final_output("workflow_a", document_number, document_id)
    output_b = load_final_output("workflow_b", document_number, document_id)
    predictions["Workflow A"][document_id] = output_a
    predictions["Workflow B"][document_id] = output_b
    completion_rows.append({
        "document_id": document_id,
        "workflow_a_complete": output_a is not None,
        "workflow_b_complete": output_b is not None,
    })

completion = pd.DataFrame(completion_rows)
assert completion["workflow_a_complete"].sum() == 5
assert completion["workflow_b_complete"].sum() == 7
print("Loaded frozen results and reference data; no API client is imported.") 

Loaded frozen results and reference data; no API client is imported.


In [2]:
# Put the main frozen results in one small table for an easy comparison.
# Completion describes technical success, while precision, recall, and F1 describe extraction quality on the five paired documents.
technical_summary = pd.DataFrame([
    {
        "workflow": "Workflow A",
        "completion": "5/9 (55.6%)",
        "logical_calls": 52,
        "runtime_seconds": 2117.296,
        "paired_precision": 0.147887,
        "paired_recall": 0.600000,
        "paired_f1": 0.237288,
    },
    {
        "workflow": "Workflow B",
        "completion": "7/9 (77.8%)",
        "logical_calls": 202,
        "runtime_seconds": 2861.814,
        "paired_precision": 0.124088,
        "paired_recall": 0.485714,
        "paired_f1": 0.197674,
    },
])
display(technical_summary)


,workflow,completion,logical_calls,runtime_seconds,paired_precision,paired_recall,paired_f1
0,Workflow A,5/9 (55.6%),52,2117.296,0.147887,0.600000,0.237288
1,Workflow B,7/9 (77.8%),202,2861.814,0.124088,0.485714,0.197674


## 2. Document-level error profile

The table below uses the exact frozen Notebook 04 matcher. The matcher code is loaded directly from the relevant frozen Notebook 04 cells to avoid introducing a second implementation. “Paired” denotes the five documents completed by both workflows; AEMET and EFE are shown as additional Workflow B-only results. These few documents are descriptive and are not used to claim a statistical correlation.

In [3]:
# Reuse the final matching functions directly from Notebook 04. 
# This avoids creating a second version of the evaluation rules for the error analysis.
notebook_04 = json.loads(NOTEBOOK_04_PATH.read_text(encoding="utf-8"))
matcher_cell_ids = {"04dc02ec", "67c890fe", "8f4039fd", "2a60e402"}
# Run the selected evaluation cells in a separate namespace with the saved data.
frozen = {
    "pd": pd,
    "math": math,
    "re": re,
    "unicodedata": unicodedata,
    "reference": reference,
    "summary": summary,
    "completion": completion,
    "predictions": predictions,
}
# Hide repeated setup output while loading the already approved matcher.
with contextlib.redirect_stdout(io.StringIO()):
    for cell in notebook_04["cells"]:
        if cell.get("id") in matcher_cell_ids:
            exec("".join(cell["source"]), frozen)

# Bring only the functions and variables needed below into this notebook.
FACT_TYPES = frozen["FACT_TYPES"]
reference_events = frozen["reference_events"]
PAIRED_DOCUMENTS = frozen["PAIRED_DOCUMENTS"]
B_ONLY_DOCUMENTS = frozen["B_ONLY_DOCUMENTS"]
prediction_facts = frozen["prediction_facts"]
match_facts = frozen["match_facts"]
align_events = frozen["align_events"]
metric_values = frozen["metric_values"]
evaluate_document = frozen["evaluate_document"]

assert [item.split(" ", 1)[0] for item in PAIRED_DOCUMENTS] == [
    "PWC", "Krausmann", "Lavenir", "ABC", "Plag"
]
assert {item.split(" ", 1)[0] for item in B_ONLY_DOCUMENTS} == {"AEMET", "EFE"}
print("Loaded the frozen Notebook 04 matching functions.")

Loaded the frozen Notebook 04 matching functions.

In [4]:
# Use short names to keep the displayed tables readable.
def short_document_name(document_id):
    return document_id.split(" ", 1)[0].replace("Lavenir", "L'Avenir")


# Count all four fact types in the reference for one document.
def total_reference_facts(document_id):
    return sum(
        len(event[fact_type])
        for event in reference_events[document_id]
        for fact_type in FACT_TYPES
    )


# Count all facts produced across every predicted event.
def total_predicted_facts(prediction):
    return sum(
        len(prediction_facts(event, fact_type))
        for event in prediction["events"]
        for fact_type in FACT_TYPES
    )


# Build one profile row for every technically complete workflow output.
# The prediction/reference ratio helps show where over-extraction is strongest.
profile_rows = []
for workflow, completion_column in [
    ("Workflow A", "workflow_a_complete"),
    ("Workflow B", "workflow_b_complete"),
]:
    for document_id in completion.loc[completion[completion_column], "document_id"]:
        prediction = predictions[workflow][document_id]
        field_rows, _ = evaluate_document(document_id, prediction)
        tp = sum(row["tp"] for row in field_rows)
        fp = sum(row["fp"] for row in field_rows)
        fn = sum(row["fn"] for row in field_rows)
        precision, recall, f1 = metric_values(tp, fp, fn)
        summary_row = summary.loc[summary["document_id"] == document_id].iloc[0]
        profile_rows.append({
            "subset": "Paired" if document_id in PAIRED_DOCUMENTS else "B-only",
            "document": short_document_name(document_id),
            "workflow": workflow,
            "word_count": int(summary_row["document_word_count"]),
            "chunks": int(summary_row["chunk_count"]),
            "reference_events": len(reference_events[document_id]),
            "predicted_events": len(prediction["events"]),
            "reference_facts": total_reference_facts(document_id),
            "predicted_facts": total_predicted_facts(prediction),
            "tp": tp,
            "fp": fp,
            "fn": fn,
            "precision": precision,
            "recall": recall,
            "f1": f1,
        })

document_error_profile = pd.DataFrame(profile_rows).sort_values(
    ["subset", "document", "workflow"],
    key=lambda column: column.map({"Paired": 0, "B-only": 1}).fillna(0)
    if column.name == "subset" else column,
)
display(document_error_profile.round({"precision": 3, "recall": 3, "f1": 3}))


,subset,document,workflow,word_count,chunks,reference_events,predicted_events,reference_facts,predicted_facts,tp,fp,fn,precision,recall,f1
3,Paired,ABC,Workflow A,706,1,1,7,11,29,7,22,4,0.241,0.636,0.350
9,Paired,ABC,Workflow B,706,1,1,20,11,67,3,64,8,0.045,0.273,0.077
1,Paired,Krausmann,Workflow A,1469,2,1,8,20,93,11,82,9,0.118,0.550,0.195
6,Paired,Krausmann,Workflow B,1469,2,1,7,20,49,10,39,10,0.204,0.500,0.290
2,Paired,L'Avenir,Workflow A,157,1,0,1,0,7,0,7,0,0.000,NaN,NaN
8,Paired,L'Avenir,Workflow B,157,1,0,0,0,0,0,0,0,NaN,NaN,NaN
0,Paired,PWC,Workflow A,484,1,1,2,4,13,3,10,1,0.231,0.750,0.353
5,Paired,PWC,Workflow B,484,1,1,1,4,10,4,6,0,0.400,1.000,0.571
4,Paired,Plag,Workflow A,1457,3,0,0,0,0,0,0,0,NaN,NaN,NaN
11,Paired,Plag,Workflow B,1457,3,0,5,0,11,0,11,0,0.000,NaN,NaN


## 3. Event fragmentation

Event IDs are bookkeeping labels, not semantic identifiers. The secondary structural analysis aligns events only to count excess structure. The cross-chunk merge in Notebook 03 merges adjacent-chunk events only when they share both a normalized evidence quotation and a hazard category. Descriptions of the same episode supported by different quotations can therefore remain separate. This is a plausible mechanism for fragmentation, but it does not prove that every excess predicted event is a duplicate.

In [5]:
# Compare reference and predicted event counts for completed documents.
# Unaligned events and their facts are reported as a structural problem
fragmentation_rows = []
for _, row in completion.iterrows():
    document_id = row["document_id"]
    for workflow, complete_column in [
        ("Workflow A", "workflow_a_complete"),
        ("Workflow B", "workflow_b_complete"),
    ]:
        if not row[complete_column]:
            continue
        predicted_events = predictions[workflow][document_id]["events"]
        alignment = align_events(reference_events[document_id], predicted_events)
        aligned_prediction_indices = {pi for _, pi, _ in alignment}
        unaligned_indices = set(range(len(predicted_events))) - aligned_prediction_indices
        unaligned_facts = sum(
            len(prediction_facts(predicted_events[index], fact_type))
            for index in unaligned_indices
            for fact_type in FACT_TYPES
        )
        fragmentation_rows.append({
            "document": short_document_name(document_id),
            "workflow": workflow,
            "reference_events": len(reference_events[document_id]),
            "predicted_events": len(predicted_events),
            "unaligned_predicted_events": len(unaligned_indices),
            "facts_in_unaligned_events": unaligned_facts,
        })

fragmentation = pd.DataFrame(fragmentation_rows)
display(fragmentation)


,document,workflow,reference_events,predicted_events,unaligned_predicted_events,facts_in_unaligned_events
0,PWC,Workflow A,1,2,1,4
1,PWC,Workflow B,1,1,0,0
2,Krausmann,Workflow A,1,8,7,40
3,Krausmann,Workflow B,1,7,6,39
4,AEMET,Workflow B,1,63,62,413
5,L'Avenir,Workflow A,0,1,1,7
6,L'Avenir,Workflow B,0,0,0,0
7,ABC,Workflow A,1,7,6,25
8,ABC,Workflow B,1,20,19,42
9,EFE,Workflow B,1,18,17,249


## 4. Qualitative false-positive analysis

The following deterministic sample contains 14 unmatched predicted facts across the requested documents. It is intentionally small and illustrates recurring error types rather than estimating their prevalence. Every row is selected from the actual unmatched predictions produced by the frozen matcher.

In [6]:
# Convert each structured fact into a short readable label for the audit tables.
def fact_text(fact_type, fact):
    if fact_type == "hazard":
        return f"[{fact['category']}] {fact['source_text']}"
    if fact_type == "location":
        name = fact.get("normalized_name") or ""
        return f"{fact['source_text']} -> {name}" if name else fact["source_text"]
    if fact_type == "time":
        normalized = fact.get("normalized_start") or ""
        return f"{fact['source_text']} -> {normalized}" if normalized else fact["source_text"]
    return f"[{fact['category']}] {fact['description']}"


# Flatten event-level lists so facts can be checked across the whole document.
def flattened_facts(events, fact_type, predicted):
    rows = []
    for event in events:
        facts = prediction_facts(event, fact_type) if predicted else event[fact_type]
        for fact in facts:
            rows.append({"event_id": event["event_id"], "fact": fact})
    return rows


# Collect predicted facts that the deterministic matcher counted as false positives.
def unmatched_prediction_rows(document_id, workflow):
    rows = []
    predicted_events = predictions[workflow][document_id]["events"]
    for fact_type in FACT_TYPES:
        references = flattened_facts(reference_events[document_id], fact_type, False)
        predicted = flattened_facts(predicted_events, fact_type, True)
        matches = match_facts(
            [item["fact"] for item in references],
            [item["fact"] for item in predicted],
            fact_type,
        )
        matched_prediction_indices = {pi for _, pi, _ in matches}
        for index, item in enumerate(predicted):
            if index not in matched_prediction_indices:
                rows.append({
                    "document_id": document_id,
                    "document": short_document_name(document_id),
                    "workflow": workflow,
                    "predicted_event_id": item["event_id"],
                    "fact_type": fact_type,
                    "predicted_fact": fact_text(fact_type, item["fact"]),
                    "evidence_quote": item["fact"]["evidence"]["quote"],
                })
    return rows


# Gather false positives from every technically complete output.
all_unmatched_predictions = []
for workflow, completion_column in [
    ("Workflow A", "workflow_a_complete"),
    ("Workflow B", "workflow_b_complete"),
]:
    for document_id in completion.loc[completion[completion_column], "document_id"]:
        all_unmatched_predictions.extend(unmatched_prediction_rows(document_id, workflow))
all_unmatched_predictions = pd.DataFrame(all_unmatched_predictions)


# Select fixed, human-readable examples without asking an LLM to judge them.
def select_fp(document, workflow, event_id, fact_type, contains, reason):
    candidates = all_unmatched_predictions[
        (all_unmatched_predictions["document"] == document)
        & (all_unmatched_predictions["workflow"] == workflow)
        & (all_unmatched_predictions["predicted_event_id"] == event_id)
        & (all_unmatched_predictions["fact_type"] == fact_type)
        & all_unmatched_predictions["predicted_fact"].str.contains(
            contains, case=False, regex=False
        )
    ]
    assert not candidates.empty, (document, workflow, event_id, fact_type, contains)
    row = candidates.iloc[0].to_dict()
    row["reason"] = reason
    return row


fp_examples = pd.DataFrame([
    select_fp("ABC", "Workflow A", "event_02", "hazard", "localized flooding",
              "The same rainfall/flood episode was split into another event; its flood category was already matched one-to-one."),
    select_fp("ABC", "Workflow A", "event_06", "impact", "orange alert active",
              "A weather-warning status was treated as an environmental consequence."),
    select_fp("ABC", "Workflow B", "event_01", "hazard", "severe thunderstorm",
              "A repeated storm-category extraction remained after the single reference storm fact was matched."),
    select_fp("Krausmann", "Workflow A", "event_01", "hazard", "tsunami it triggered",
              "Repeated tsunami wording was categorized as flood, while the reference category is other; duplicate mentions also remained."),
    select_fp("Krausmann", "Workflow B", "event_03", "hazard", "20% probability",
              "A projected earthquake probability/background statement was treated as an actual hazard event."),
    select_fp("Krausmann", "Workflow B", "event_04", "hazard", "low levels of earthquake forces",
              "A statement about low earthquake forces was incorrectly represented as a flood hazard."),
    select_fp("Plag", "Workflow B", "event_01", "hazard", "submarine landslides",
              "General discussion of landslide risk was treated as a concrete event in a reference no-event document."),
    select_fp("Plag", "Workflow B", "event_02", "impact", "reduce impact on human lives",
              "A hypothetical benefit of a warning system was treated as a concrete human impact."),
    select_fp("Plag", "Workflow B", "event_05", "impact", "major disasters",
              "A generic statement about drought consequences was turned into a vague economic impact."),
    select_fp("AEMET", "Workflow B", "event_33", "hazard", "At night, yellow warning",
              "A precipitation warning was split into another event and normalized to the incorrect extreme_heat category."),
    select_fp("AEMET", "Workflow B", "event_60", "hazard", "During the 1982 event",
              "A historical comparison event was extracted and assigned the incorrect extreme_heat category."),
    select_fp("EFE", "Workflow B", "event_13", "impact", "1,440 incidents",
              "An operational incident count was classified as a human-health impact."),
    select_fp("EFE", "Workflow B", "event_14", "hazard", "the DANA storm",
              "The same DANA wording was extracted repeatedly after the reference storm fact was already matched."),
    select_fp("EFE", "Workflow B", "event_14", "hazard", "worst storm of the century",
              "Another description of the same episode remained in a separate fragmented event."),
])

display(fp_examples[[
    "document", "workflow", "predicted_event_id", "fact_type",
    "predicted_fact", "evidence_quote", "reason",
]])

observed_fp_categories = pd.Series([
    "duplicate/repeated extraction",
    "same episode fragmented into another event",
    "background, projected, or historical material treated as an event",
    "incorrect normalized category",
    "warning or generic statement treated as a concrete impact/event",
], name="observed_error_category").to_frame()
display(observed_fp_categories)

,document,workflow,predicted_event_id,fact_type,predicted_fact,evidence_quote,reason
0,ABC,Workflow A,event_02,hazard,"[flood] localized flooding in garages, basemen...","localized flooding in garages, basements, grou...",The same rainfall/flood episode was split into...
1,ABC,Workflow A,event_06,impact,[environmental] orange alert active,The State Meteorological Agency (Aemet) mainta...,A weather-warning status was treated as an env...
2,ABC,Workflow B,event_01,hazard,[storm] severe thunderstorm,severe thunderstorm,A repeated storm-category extraction remained ...
3,Krausmann,Workflow A,event_01,hazard,[flood] tsunami it triggered both reached unex...,the tsunami it triggered both reached unexpect...,Repeated tsunami wording was categorized as fl...
4,Krausmann,Workflow B,event_03,hazard,[flood] tsunami earthquakes had been expected ...,tsunami earthquakes had been expected to occur...,A projected earthquake probability/background ...
5,Krausmann,Workflow B,event_04,hazard,[flood] The Cosmo Oil refinery is located in T...,The Cosmo Oil refinery is located in Tokyo Bay...,A statement about low earthquake forces was in...
6,Plag,Workflow B,event_01,hazard,"[other] For submarine landslides, characterizi...","For submarine landslides, characterizing the f...",General discussion of landslide risk was treat...
7,Plag,Workflow B,event_02,impact,[human_health] reduce impact on human lives,"However, for larger submarine landslides, whic...",A hypothetical benefit of a warning system was...
8,Plag,Workflow B,event_05,impact,[economic] major disasters,"Extreme droughts have caused major disasters, ...",A generic statement about drought consequences...
9,AEMET,Workflow B,event_33,hazard,"[extreme_heat] At night, yellow warning of pre...","At night, yellow warning of precipitation of 2...",A precipitation warning was split into another...


,observed_error_category
0,duplicate/repeated extraction
1,same episode fragmented into another event
2,"background, projected, or historical material ..."
3,incorrect normalized category
4,warning or generic statement treated as a conc...


## 5. Representative false negatives

This sample contains eight unmatched reference facts from the paired subset. “Related prediction rejected” means that a prediction expressed related content but failed at least one frozen matching condition. “Absent” means no prediction represented that fact in the corresponding field. These examples do not change the matcher.

In [7]:
# Collect reference facts that were not matched and therefore count as false negatives.
def unmatched_reference_rows(document_id, workflow):
    rows = []
    predicted_events = predictions[workflow][document_id]["events"]
    for fact_type in FACT_TYPES:
        references = flattened_facts(reference_events[document_id], fact_type, False)
        predicted = flattened_facts(predicted_events, fact_type, True)
        matches = match_facts(
            [item["fact"] for item in references],
            [item["fact"] for item in predicted],
            fact_type,
        )
        matched_reference_indices = {ri for ri, _, _ in matches}
        for index, item in enumerate(references):
            if index not in matched_reference_indices:
                rows.append({
                    "document_id": document_id,
                    "document": short_document_name(document_id),
                    "workflow": workflow,
                    "fact_type": fact_type,
                    "reference_fact": fact_text(fact_type, item["fact"]),
                    "evidence_quote": item["fact"]["evidence_quote"],
                })
    return rows


# Use the paired subset so missing facts are compared fairly for both workflows.
all_unmatched_references = []
for workflow in ["Workflow A", "Workflow B"]:
    for document_id in PAIRED_DOCUMENTS:
        all_unmatched_references.extend(unmatched_reference_rows(document_id, workflow))
all_unmatched_references = pd.DataFrame(all_unmatched_references)


# Select fixed examples and explain whether the fact was missing or mismatched.
def select_fn(document, workflow, fact_type, contains, prediction_status):
    candidates = all_unmatched_references[
        (all_unmatched_references["document"] == document)
        & (all_unmatched_references["workflow"] == workflow)
        & (all_unmatched_references["fact_type"] == fact_type)
        & all_unmatched_references["reference_fact"].str.contains(
            contains, case=False, regex=False
        )
    ]
    assert not candidates.empty, (document, workflow, fact_type, contains)
    row = candidates.iloc[0].to_dict()
    row["prediction_status"] = prediction_status
    return row


fn_examples = pd.DataFrame([
    select_fn("PWC", "Workflow A", "impact", "insurance industry paying out",
              "Related prediction existed, but its qualifier signature omitted 'between'."),
    select_fn("Krausmann", "Workflow A", "hazard", "Great East Japan earthquake",
              "Related earthquake/tsunami wording existed but was assigned flood rather than other."),
    select_fn("Krausmann", "Workflow A", "impact", "2,636 people missing",
              "Related prediction existed but used human_health rather than the reference category other."),
    select_fn("Krausmann", "Workflow B", "impact", "evacuation of 1,142 residents",
              "Related prediction existed but used a different impact category."),
    select_fn("Krausmann", "Workflow B", "impact", "2-year business interruption",
              "Related prediction existed, but added wording reduced lexical similarity below the frozen threshold."),
    select_fn("ABC", "Workflow A", "impact", "flight delays",
              "Related prediction existed, but the longer description failed the conservative impact matcher."),
    select_fn("ABC", "Workflow A", "impact", "road subsidence",
              "Absent from the predicted impact list."),
    select_fn("ABC", "Workflow B", "hazard", "localized flooding",
              "Related wording existed but was normalized as storm rather than flood."),
])

display(fn_examples[[
    "document", "workflow", "fact_type", "reference_fact",
    "evidence_quote", "prediction_status",
]])

,document,workflow,fact_type,reference_fact,evidence_quote,prediction_status
0,PWC,Workflow A,impact,[economic] insurance industry paying out betwe...,Our current estimate of the damage caused by S...,"Related prediction existed, but its qualifier ..."
1,Krausmann,Workflow A,hazard,[other] Great East Japan earthquake,The Great East Japan earthquake and the tsunam...,Related earthquake/tsunami wording existed but...
2,Krausmann,Workflow A,impact,"[other] 2,636 people missing",A recent report by the Japanese National Polic...,Related prediction existed but used human_heal...
3,Krausmann,Workflow B,impact,"[other] evacuation of 1,142 residents",The fires and explosions at the Cosmo Oil refi...,Related prediction existed but used a differen...
4,Krausmann,Workflow B,impact,[economic] 2-year business interruption,"Overall, the refinery suffered a 2-year busine...","Related prediction existed, but added wording ..."
5,ABC,Workflow A,impact,[transport] flight delays of approximately 15 ...,"At Malaga Airport, a severe thunderstorm force...","Related prediction existed, but the longer des..."
6,ABC,Workflow A,impact,[infrastructure] road subsidence,There have also been some traffic jams and roa...,Absent from the predicted impact list.
7,ABC,Workflow B,hazard,[flood] localized flooding,"In addition, the rainfall has caused localized...",Related wording existed but was normalized as ...


## 6. Evidence support versus correctness

Evidence support is an exact provenance check only: the quotation must occur verbatim in the cleaned Markdown. It does not establish that the fact is relevant to the target event, that the event qualifies, or that category and scope agree with the reference.

In [8]:
# Check whether each evidence quote appears exactly in its cleaned document.
# This checks provenance only. it does not prove that the extracted fact is correct.
def evidence_rows(workflow, document_ids):
    rows = []
    for document_id in document_ids:
        source = document_texts[document_id]
        for event in predictions[workflow][document_id]["events"]:
            for fact_type in FACT_TYPES:
                for fact in prediction_facts(event, fact_type):
                    quote = fact["evidence"]["quote"]
                    rows.append({
                        "workflow": workflow,
                        "document_id": document_id,
                        "supported": quote in source,
                    })
    return pd.DataFrame(rows)


# First compare evidence support on the same five paired documents.
paired_evidence = pd.concat([
    evidence_rows("Workflow A", PAIRED_DOCUMENTS),
    evidence_rows("Workflow B", PAIRED_DOCUMENTS),
], ignore_index=True)
# Then summarize all technically complete outputs for each workflow.
all_complete_evidence = pd.concat([
    evidence_rows(
        workflow,
        completion.loc[completion[completion_column], "document_id"].tolist(),
    )
    for workflow, completion_column in [
        ("Workflow A", "workflow_a_complete"),
        ("Workflow B", "workflow_b_complete"),
    ]
], ignore_index=True)


# Calculate the number and percentage of verbatim-supported quotations.
def summarize_evidence(rows, subset):
    output = []
    for workflow, group in rows.groupby("workflow", sort=False):
        total = len(group)
        supported = int(group["supported"].sum())
        output.append({
            "subset": subset,
            "workflow": workflow,
            "supported": supported,
            "total": total,
            "supported_rate": supported / total,
        })
    return output


evidence_summary = pd.DataFrame(
    summarize_evidence(paired_evidence, "Paired")
    + summarize_evidence(all_complete_evidence, "All complete")
)
display(evidence_summary.round({"supported_rate": 4}))

assert evidence_summary[["supported", "total"]].values.tolist() == [
    [138, 142], [131, 137], [138, 142], [772, 823]
]


,subset,workflow,supported,total,supported_rate
0,Paired,Workflow A,138,142,0.9718
1,Paired,Workflow B,131,137,0.9562
2,All complete,Workflow A,138,142,0.9718
3,All complete,Workflow B,772,823,0.9380


## 7. No-event documents

L'Avenir and Plag test whether the workflows avoid constructing events from non-qualifying material. Plag is a broad foreword about geohazards rather than an account of one target episode. Workflow B converted general landslide risk, historical or current volcanic context, and hypothetical drought consequences into five events, even though the frozen reference classifies the document as no-event.

In [9]:
# Inspect the two documents annotated as containing no qualifying event.
no_event_results = pd.DataFrame([
    {"document": "L'Avenir", "workflow": "Workflow A", "predicted_events": 1, "predicted_facts": 7, "correct": False},
    {"document": "L'Avenir", "workflow": "Workflow B", "predicted_events": 0, "predicted_facts": 0, "correct": True},
    {"document": "Plag", "workflow": "Workflow A", "predicted_events": 0, "predicted_facts": 0, "correct": True},
    {"document": "Plag", "workflow": "Workflow B", "predicted_events": 5, "predicted_facts": 11, "correct": False},
])
display(no_event_results)

# Show the unsupported event structure produced for Plag by Workflow B.
plag_id = next(document for document in reference_events if document.startswith("Plag "))
plag_rows = []
for event in predictions["Workflow B"][plag_id]["events"]:
    for fact_type in FACT_TYPES:
        for fact in prediction_facts(event, fact_type):
            plag_rows.append({
                "event_id": event["event_id"],
                "fact_type": fact_type,
                "prediction": fact_text(fact_type, fact),
            })
display(pd.DataFrame(plag_rows))

,document,workflow,predicted_events,predicted_facts,correct
0,L'Avenir,Workflow A,1,7,False
1,L'Avenir,Workflow B,0,0,True
2,Plag,Workflow A,0,0,True
3,Plag,Workflow B,5,11,False


,event_id,fact_type,prediction
0,event_01,hazard,"[other] For submarine landslides, characterizi..."
1,event_02,hazard,"[other] For coastal landslides, the warning ti..."
2,event_02,impact,[human_health] reduce impact on human lives
3,event_03,hazard,[other] Volcanoes exhibit a wide range of haza...
4,event_03,location,Santorini is the location of one of the bigges...
5,event_04,hazard,[other] The Kolumbo subsea volcano to the Nort...
6,event_04,location,The Kolumbo subsea volcano to the Northeast of...
7,event_04,impact,[environmental] impacting the biological envir...
8,event_05,hazard,[other] Extreme droughts have caused major dis...
9,event_05,impact,[economic] major disasters


## 8. Workflow A versus Workflow B

The comparison below separates extraction quality, technical robustness, and efficiency. Ratios describe only this frozen experiment.

In [10]:
# Compare quality, technical completion, calls, and runtime side by side.
# The ratios describe this frozen experiment and are not general claims.
tradeoff = pd.DataFrame([
    {"measure": "Paired F1", "Workflow A": 0.237288, "Workflow B": 0.197674, "B/A ratio": 0.197674 / 0.237288},
    {"measure": "Completion rate", "Workflow A": 0.556, "Workflow B": 0.778, "B/A ratio": 0.778 / 0.556},
    {"measure": "Logical calls", "Workflow A": 52, "Workflow B": 202, "B/A ratio": 202 / 52},
    {"measure": "Runtime (seconds)", "Workflow A": 2117.296, "Workflow B": 2861.814, "B/A ratio": 2861.814 / 2117.296},
])
display(tradeoff.round(3))

print(f"B/A logical-call ratio: {202 / 52:.3f}")
print(f"B/A runtime ratio: {2861.814 / 2117.296:.3f}")

,measure,Workflow A,Workflow B,B/A ratio
0,Paired F1,0.237,0.198,0.833
1,Completion rate,0.556,0.778,1.399
2,Logical calls,52.000,202.000,3.885
3,Runtime (seconds),2117.296,2861.814,1.352


B/A logical-call ratio: 3.885
B/A runtime ratio: 1.352
